In [1]:
import os
import warnings
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 14
})

os.makedirs("../plots", exist_ok=True)
os.makedirs("../data", exist_ok=True)


In [2]:
df_orig = pd.read_csv("../data/StudentsPerformance.csv")
df_dirty = pd.read_csv("../data/StudentsPerformance_dirty.csv")

print(f"Original Benchmark Shape: {df_orig.shape}")
print(f"Working Dirty Dataset Shape: {df_dirty.shape}")
display(df_dirty.head())


Original Benchmark Shape: (1000, 8)
Working Dirty Dataset Shape: (1010, 8)


,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72.0,72.0,74.0
1,female,group C,some college,standard,completed,69.0,90.0,88.0
2,female,group B,master's degree,standard,none,90.0,95.0,93.0
3,male,group A,associate's degree,free/reduced,none,47.0,57.0,44.0
4,male,group C,some college,standard,NONE,76.0,78.0,75.0


In [3]:
quality_audit = pd.DataFrame({
    "Data Type": df_dirty.dtypes.astype(str),
    "Null Count": df_dirty.isnull().sum(),
    "Null Pct (%)": (df_dirty.isnull().mean() * 100).round(2),
    "Unique Values": df_dirty.nunique()
})
display(quality_audit)
print("Exact duplicate rows detected:", df_dirty.duplicated().sum())


,Data Type,Null Count,Null Pct (%),Unique Values
gender,str,0,0.00,4
race/ethnicity,str,0,0.00,7
parental level of education,str,0,0.00,6
lunch,str,0,0.00,4
test preparation course,str,0,0.00,4
math score,str,50,4.95,84
reading score,str,50,4.95,75
writing score,str,50,4.95,79


Exact duplicate rows detected: 8


In [4]:
clean_df = df_dirty.copy()

# Categorical text standardization
cat_cols = ["gender", "race/ethnicity", "parental level of education", "lunch", "test preparation course"]
for col in cat_cols:
    clean_df[col] = clean_df[col].astype(str).str.strip().str.lower()

# Numeric coercion for score features
score_cols = ["math score", "reading score", "writing score"]
for col in score_cols:
    clean_df[col] = pd.to_numeric(clean_df[col], errors="coerce")

# Range validation [0, 100]
for col in score_cols:
    invalid_mask = (clean_df[col] < 0) | (clean_df[col] > 100)
    clean_df.loc[invalid_mask, col] = np.nan

# Deduplication
clean_df = clean_df.drop_duplicates().reset_index(drop=True)

# Median imputation
medians = clean_df[score_cols].median()
clean_df[score_cols] = clean_df[score_cols].fillna(medians)

# Feature engineering
clean_df["total_score"] = clean_df[score_cols].sum(axis=1)
clean_df["average_score"] = clean_df["total_score"] / 3

# Export cleaned production dataset
clean_df.to_csv("../data/StudentsPerformance_cleaned.csv", index=False)
print("Cleaned dataset shape:", clean_df.shape)
print("Remaining null values:", clean_df.isnull().sum().sum())
print("Remaining duplicate rows:", clean_df.duplicated().sum())

# Validation against the original benchmark (the original is only used to CHECK the cleaning, not to clean)
print("\nPost-cleaning category counts:")
for col in cat_cols:
    print(f"  {col}: {clean_df[col].value_counts().to_dict()}")
print(f"\nRow count: cleaned = {len(clean_df)}, original = {len(df_orig)} (difference = {len(clean_df) - len(df_orig)})")
comparison = pd.DataFrame({
    "Original mean": df_orig[score_cols].mean(),
    "Cleaned mean": clean_df[score_cols].mean()
}).round(2)
comparison["Difference"] = (comparison["Cleaned mean"] - comparison["Original mean"]).round(2)
display(comparison)
print("Reading-Writing r: original =", round(df_orig["reading score"].corr(df_orig["writing score"]), 3),
      "| cleaned =", round(clean_df["reading score"].corr(clean_df["writing score"]), 3))


Cleaned dataset shape: (1001, 10)
Remaining null values: 0
Remaining duplicate rows: 0

Post-cleaning category counts:
  gender: {'female': 518, 'male': 483}
  race/ethnicity: {'group c': 320, 'group d': 262, 'group b': 190, 'group e': 140, 'group a': 89}
  parental level of education: {'some college': 226, "associate's degree": 222, 'high school': 196, 'some high school': 180, "bachelor's degree": 118, "master's degree": 59}
  lunch: {'standard': 646, 'free/reduced': 355}
  test preparation course: {'none': 643, 'completed': 358}

Row count: cleaned = 1001, original = 1000 (difference = 1)


,Original mean,Cleaned mean,Difference
math score,66.09,65.89,-0.20
reading score,69.17,69.27,0.10
writing score,68.05,68.06,0.01


Reading-Writing r: original = 0.955 | cleaned = 0.904


In [5]:
analysis_cols = score_cols + ["total_score", "average_score"]

descriptive_table = pd.DataFrame({
    "Mean": clean_df[analysis_cols].mean(),
    "Std Dev": clean_df[analysis_cols].std(),
    "Median": clean_df[analysis_cols].median(),
    "IQR": clean_df[analysis_cols].quantile(0.75) - clean_df[analysis_cols].quantile(0.25),
    "Min": clean_df[analysis_cols].min(),
    "Max": clean_df[analysis_cols].max(),
    "Skewness": clean_df[analysis_cols].skew()
})
display(descriptive_table.round(2))


,Mean,Std Dev,Median,IQR,Min,Max,Skewness
math score,65.89,14.78,66.00,19.0,0.0,100.0,-0.29
reading score,69.27,14.24,70.00,19.0,17.0,100.0,-0.28
writing score,68.06,14.70,69.00,20.0,10.0,100.0,-0.31
total_score,203.22,40.86,206.00,54.0,27.0,300.0,-0.32
average_score,67.74,13.62,68.67,18.0,9.0,100.0,-0.32


In [6]:
plt.figure(figsize=(8.5, 5.5))
gender_data = clean_df.melt(id_vars="gender", value_vars=score_cols, var_name="Subject", value_name="Score")
gender_data["Subject"] = gender_data["Subject"].str.title()
gender_data["gender"] = gender_data["gender"].str.title()

ax1 = sns.barplot(
    data=gender_data,
    x="Subject",
    y="Score",
    hue="gender",
    palette=["#e07a5f", "#3d5a80"],
    errorbar=None,
    edgecolor="black",
    width=0.6
)
for p in ax1.patches:
    h = p.get_height()
    if h > 0:
        ax1.annotate(f"{h:.1f}", (p.get_x() + p.get_width() / 2., h / 2),
                     ha="center", va="center", color="white", weight="bold", fontsize=11)

plt.title("Plot 1: Mean Subject Scores by Gender", weight="bold", pad=15)
plt.xlabel("Academic Subject")
plt.ylabel("Mean Score (0–100)")
plt.ylim(0, 100)
plt.legend(title="Gender", frameon=True)
plt.tight_layout()
plt.savefig("../plots/plot1_bar_chart_gender.png", dpi=300, bbox_inches="tight")
plt.show()


In [7]:
r_rw = clean_df["reading score"].corr(clean_df["writing score"])
plt.figure(figsize=(8, 5.5))
sns.regplot(
    data=clean_df,
    x="reading score",
    y="writing score",
    scatter_kws={"alpha": 0.45, "color": "#2a9d8f", "edgecolor": "none"},
    line_kws={"color": "#d62828", "linewidth": 2, "label": f"Linear Fit (r = {r_rw:.3f})"}
)
plt.title("Plot 2: Reading vs. Writing Examination Scores", weight="bold", pad=15)
plt.xlabel("Reading Score (0–100)")
plt.ylabel("Writing Score (0–100)")
plt.xlim(10, 105)
plt.ylim(5, 105)
plt.legend(frameon=True, loc="upper left")
plt.tight_layout()
plt.savefig("../plots/plot2_scatter_reading_writing.png", dpi=300, bbox_inches="tight")
plt.show()


In [8]:
plt.figure(figsize=(7, 5.5))
corr_df = clean_df[score_cols].corr()
corr_df.columns = ["Math", "Reading", "Writing"]
corr_df.index = ["Math", "Reading", "Writing"]

sns.heatmap(
    corr_df,
    annot=True,
    cmap="Blues",
    fmt=".3f",
    linewidths=1,
    square=True,
    cbar_kws={"shrink": 0.82},
    annot_kws={"size": 11, "weight": "bold"}
)
plt.title("Plot 3: Pearson Correlation Matrix of Subject Scores", weight="bold", pad=15)
plt.tight_layout()
plt.savefig("../plots/plot3_heatmap_correlation.png", dpi=300, bbox_inches="tight")
plt.show()


In [9]:
test_records = []

for col in score_cols:
    f_scores = clean_df.loc[clean_df["gender"] == "female", col]
    m_scores = clean_df.loc[clean_df["gender"] == "male", col]
    
    t_val, p_val = ttest_ind(f_scores, m_scores, equal_var=False)
    
    n_f, n_m = len(f_scores), len(m_scores)
    s_f, s_m = f_scores.std(), m_scores.std()
    s_pooled = np.sqrt(((n_f - 1) * s_f**2 + (n_m - 1) * s_m**2) / (n_f + n_m - 2))
    d_val = (f_scores.mean() - m_scores.mean()) / s_pooled
    
    test_records.append({
        "Subject": col.title(),
        "Female Mean": f_scores.mean(),
        "Male Mean": m_scores.mean(),
        "Difference (F - M)": f_scores.mean() - m_scores.mean(),
        "T-statistic": t_val,
        "Raw P-value": p_val,
        "Cohen's d": d_val
    })

results_table = pd.DataFrame(test_records)
results_table["Bonferroni P-adj"] = multipletests(results_table["Raw P-value"], method="bonferroni")[1]
results_table["Significant (alpha=0.05)"] = results_table["Bonferroni P-adj"] < 0.05

print("GENDER COMPARISON TEST RESULTS (WELCH'S T-TEST & COHEN'S D):")
display_table = results_table.copy()
for c in ["Raw P-value", "Bonferroni P-adj"]:
    display_table[c] = display_table[c].apply(lambda p: "< 0.001" if p < 0.001 else f"{p:.3f}")
display(display_table.round(4))


GENDER COMPARISON TEST RESULTS (WELCH'S T-TEST & COHEN'S D):


,Subject,Female Mean,Male Mean,Difference (F - M),T-statistic,Raw P-value,Cohen's d,Bonferroni P-adj,Significant (alpha=0.05)
0,Math Score,63.5656,68.3810,-4.8153,-5.2376,< 0.001,-0.3301,< 0.001,True
1,Reading Score,72.5946,65.7039,6.8907,7.8871,< 0.001,0.4984,< 0.001,True
2,Writing Score,72.0714,63.7681,8.3033,9.3190,< 0.001,0.5884,< 0.001,True
